# Forge V2 - Smartphone Price Prediction


Objective

Build a production-ready machine learning pipeline for smartphone price prediction.

*Goals*
- Improve V1 model performance
- Engineer better features
- Compare multiple ML models
- Build a reusable preprocessing pipeline
- Export a production ready model

In [2]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Baseline Model
from sklearn.linear_model import LinearRegression

# Settings
pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("data_smartphone.csv")
df.head()

,brand_name,Name,Price,RAM,OS,storage,Battery_cap,has_fast_charging,has_fingerprints,has_nfc,has_5g,processor_brand,num_core,primery_rear_camera,Num_Rear_Cameras,primery_front_camera,num_front_camera,display_size(inch),refresh_rate(hz),display_types
0,vivo,vivo v50,34999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,50.0,1,6.77,120.0,amoled display
1,realme,realme p3 pro,21999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,16.0,1,6.83,120.0,amoled display
2,realme,realme 14 pro plus,27999,8.0,android,128.0,6000,Yes,Yes,No,Yes,snapdragon,8.0,50.0,3,32.0,1,6.83,120.0,oled display
3,samsung,samsung galaxy s25 ultra,129999,12.0,android,256.0,5000,Yes,Yes,Yes,Yes,snapdragon,8.0,200.0,4,12.0,1,6.90,120.0,amoled display
4,vivo,vivo t3 pro,22999,8.0,android,128.0,5500,Yes,Yes,No,Yes,snapdragon,8.0,50.0,2,16.0,1,6.77,120.0,amoled display


In [4]:
print(df.shape)
df.info()

(3260, 20)
<class 'pandas.DataFrame'>
RangeIndex: 3260 entries, 0 to 3259
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   brand_name            3260 non-null   str    
 1   Name                  3260 non-null   str    
 2   Price                 3260 non-null   int64  
 3   RAM                   3260 non-null   float64
 4   OS                    3260 non-null   str    
 5   storage               3260 non-null   float64
 6   Battery_cap           3260 non-null   int64  
 7   has_fast_charging     3260 non-null   str    
 8   has_fingerprints      2534 non-null   str    
 9   has_nfc               2534 non-null   str    
 10  has_5g                2534 non-null   str    
 11  processor_brand       3260 non-null   str    
 12  num_core              3085 non-null   float64
 13  primery_rear_camera   3260 non-null   float64
 14  Num_Rear_Cameras      3260 non-null   int64  
 15  primery_front_camera 

In [5]:
df["Name"].head(30)

0                         vivo v50
1                    realme p3 pro
2               realme 14 pro plus
3         samsung galaxy s25 ultra
4                      vivo t3 pro
5          motorola edge 50 fusion
6                         moto g85
7                      oneplus 13r
8                      poco x7 pro
9             oneplus nord ce 4 5g
10                        vivo v40
11           samsung galaxy m35 5g
12                      oneplus 13
13                      iqoo 13 5g
14    xiaomi redmi note 14 pro+ 5g
15                  oneplus nord 4
16           samsung galaxy f06 5g
17           samsung galaxy a35 5g
18         xiaomi redmi note 14 5g
19         motorola edge 50 pro 5g
20                   vivo x200 pro
21                       vivo v40e
22              samsung galaxy s25
23                    iqoo z9s pro
24                        iqoo z9s
25            motorola edge 50 neo
26                        vivo t3x
27        samsung galaxy s24 ultra
28                  

In [7]:
df["num_core"].info()
df["num_core"].value_counts(dropna=False)

<class 'pandas.Series'>
RangeIndex: 3260 entries, 0 to 3259
Series name: num_core
Non-Null Count  Dtype  
--------------  -----  
3085 non-null   float64
dtypes: float64(1)
memory usage: 25.6 KB


num_core
8.0     2332
4.0      613
NaN      175
6.0      103
2.0       26
10.0       9
1.0        2
Name: count, dtype: int64

In [8]:
df[df["num_core"].isna()]

,brand_name,Name,Price,RAM,OS,storage,Battery_cap,has_fast_charging,has_fingerprints,has_nfc,has_5g,processor_brand,num_core,primery_rear_camera,Num_Rear_Cameras,primery_front_camera,num_front_camera,display_size(inch),refresh_rate(hz),display_types
87,google,google pixel 8a,37999,8.00,android,128.0,4492,Yes,Yes,Yes,Yes,google,NaN,64.0,2,13.0,1,6.1,120.0,oled display
162,google,google pixel 8,49999,8.00,android,128.0,4575,Yes,Yes,Yes,Yes,google,NaN,50.0,2,10.5,1,6.2,120.0,oled display
284,google,google pixel 8 pro,101999,12.00,android,128.0,5050,Yes,Yes,Yes,Yes,google,NaN,50.0,3,10.5,1,6.7,120.0,oled display
487,Other,reliance jiophone prima 2 4g,2799,0.50,other,4.0,2000,No,NaN,NaN,NaN,snapdragon,NaN,0.3,1,0.3,1,2.4,NaN,tft display
727,google,google pixel 8a 256gb,59999,8.00,android,256.0,4492,Yes,Yes,Yes,Yes,google,NaN,64.0,2,13.0,1,6.1,120.0,oled display
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3142,panasonic,panasonic eluga l 4g,9792,1.00,android,8.0,2000,No,NaN,NaN,NaN,snapdragon,NaN,8.0,1,5.0,1,5.0,NaN,lcd display
3150,intex,intex aqua 4g strong,4999,0.75,android,4.0,1700,No,NaN,NaN,NaN,mediatek,NaN,2.0,1,0.3,1,4.0,NaN,tft display
3170,panasonic,panasonic eluga tapp,6500,2.00,android,16.0,2800,No,Yes,No,No,mediatek,NaN,8.0,1,5.0,1,5.0,NaN,lcd display
3193,panasonic,panasonic eluga i2 activ,7490,1.00,android,16.0,2200,No,NaN,NaN,NaN,mediatek,NaN,8.0,1,5.0,1,5.0,NaN,lcd display


In [9]:
df['num_core'] = df['num_core'].fillna(df['num_core'].median())

In [10]:
print(df.isna().sum())  

brand_name                 0
Name                       0
Price                      0
RAM                        0
OS                         0
storage                    0
Battery_cap                0
has_fast_charging          0
has_fingerprints         726
has_nfc                  726
has_5g                   726
processor_brand            0
num_core                   0
primery_rear_camera        0
Num_Rear_Cameras           0
primery_front_camera       0
num_front_camera           0
display_size(inch)         0
refresh_rate(hz)        1731
display_types              0
dtype: int64


In [13]:
print(f"Duplicate rows: {df.duplicated().sum()}")
df.describe().T

Duplicate rows: 0


,count,mean,std,min,25%,50%,75%,max
Price,3260.0,20181.384356,24145.388368,2500.00,7490.0,11999.000,21999.00,200999.00
RAM,3260.0,5.065874,3.256896,0.25,3.0,4.000,8.00,24.00
storage,3260.0,112.040893,126.893532,0.31,32.0,64.000,128.00,1024.00
Battery_cap,3260.0,4163.485583,1312.404904,1100.00,3007.5,4500.000,5000.00,22000.00
num_core,3260.0,7.138037,1.649559,1.00,8.0,8.000,8.00,10.00
primery_rear_camera,3260.0,32.655828,29.397695,0.30,12.0,16.000,50.00,200.00
Num_Rear_Cameras,3260.0,2.076994,0.990856,1.00,1.0,2.000,3.00,5.00
primery_front_camera,3260.0,12.555767,10.564795,0.30,5.0,8.000,16.00,60.00
num_front_camera,3260.0,1.026994,0.162090,1.00,1.0,1.000,1.00,2.00
display_size(inch),3260.0,6.097110,0.741478,2.40,5.5,6.455,6.67,8.03


In [18]:
columns_to_drop = [
    "refresh_rate(hz)",
    "has_5g",
    "has_nfc",
    "has_fingerprints"
]

df = df.drop(columns=columns_to_drop)

In [20]:
df = df.rename(columns={
    "Name": "phone_name",
    "Price": "price",
    "RAM": "ram",
    "OS": "os",
    "Battery_cap": "battery_capacity",
    "has_fingerprints": "has_fingerprint",
    "num_core": "num_cores",
    "primery_rear_camera": "primary_rear_camera",
    "Num_Rear_Cameras": "num_rear_cameras",
    "primery_front_camera": "primary_front_camera",
    "num_front_camera": "num_front_cameras",
    "display_size(inch)": "display_size",
    "refresh_rate(hz)": "refresh_rate",
    "display_types": "display_type"
})

In [21]:
df.to_csv("cleaned_smartphones.csv", index=False)